# 🚀 iOS App Development Workflow

**Visual step-by-step app building using nvidia-cli tools**

This notebook provides a guided workflow for:
1. Setting up your project
2. Reading and analyzing code
3. Making edits and fixes
4. Building and testing
5. Iterating until complete

---
## 📦 Setup: Import nvidia-cli Tools

In [ ]:
import subprocess
import json
import os
from pathlib import Path
from IPython.display import display, Markdown, HTML, Image

# nvidia-cli project path
NVIDIA_CLI_PATH = "/Users/home/Documents/nvidia-cli"

# Current iOS project (change this)
PROJECT_PATH = "/Users/home/Documents/iOS/3-2-1-Liftoff/3-2-1-Liftoff"

def bash(command: str, cwd: str = None, timeout: int = 120) -> str:
    """Execute a shell command."""
    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=cwd or PROJECT_PATH,
            capture_output=True,
            text=True,
            timeout=timeout,
            env={**os.environ, "PATH": f"/opt/homebrew/bin:/usr/local/bin:{os.environ.get('PATH', '')}"}
        )
        output = result.stdout
        if result.stderr:
            output += f"\n[stderr] {result.stderr}"
        return output or "[Command completed with no output]"
    except subprocess.TimeoutExpired:
        return "[Command timed out]"
    except Exception as e:
        return f"[Error] {str(e)}"

def file_read(path: str) -> str:
    """Read a file's contents."""
    full_path = path if path.startswith("/") else os.path.join(PROJECT_PATH, path)
    try:
        with open(full_path, 'r') as f:
            return f.read()
    except Exception as e:
        return f"[Error] {str(e)}"

def file_write(path: str, content: str) -> str:
    """Write content to a file."""
    full_path = path if path.startswith("/") else os.path.join(PROJECT_PATH, path)
    try:
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, 'w') as f:
            f.write(content)
        return f"✅ Wrote {len(content)} chars to {path}"
    except Exception as e:
        return f"[Error] {str(e)}"

def list_files(path: str = ".", pattern: str = "*") -> str:
    """List files in a directory."""
    full_path = path if path.startswith("/") else os.path.join(PROJECT_PATH, path)
    try:
        items = sorted(Path(full_path).glob(pattern))
        result = []
        for item in items:
            prefix = "📁" if item.is_dir() else "📄"
            result.append(f"{prefix} {item.name}")
        return "\n".join(result) or "[No files found]"
    except Exception as e:
        return f"[Error] {str(e)}"

def show(text: str):
    """Display text with syntax highlighting."""
    display(Markdown(f"```\n{text}\n```"))

def show_swift(code: str):
    """Display Swift code with syntax highlighting."""
    display(Markdown(f"```swift\n{code}\n```"))

def screenshot() -> str:
    """Take a screenshot of the iOS simulator."""
    import datetime
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/sim_{timestamp}.png"
    bash(f"xcrun simctl io booted screenshot '{path}'")
    display(Image(filename=path, width=300))
    return path

print(f"✅ Tools loaded")
print(f"📁 Project: {PROJECT_PATH}")

---
## 1️⃣ Set Project Path

Change `PROJECT_PATH` above to your iOS project, then run this cell to verify:

In [ ]:
# Verify project exists and show structure
print(f"📁 Project: {PROJECT_PATH}\n")
print(list_files(PROJECT_PATH))

---
## 2️⃣ Explore Project Structure

In [ ]:
# Find all Swift files
swift_files = bash(f"find {PROJECT_PATH} -name '*.swift' -not -path '*/.*' | head -30")
print("📄 Swift Files:")
print(swift_files)

---
## 3️⃣ Read a File

In [ ]:
# Read a specific file (change the path)
FILE_TO_READ = "ContentView.swift"  # Change this

content = file_read(FILE_TO_READ)
show_swift(content)

---
## 4️⃣ Search for Code Patterns

In [ ]:
# Search for a pattern in Swift files
SEARCH_PATTERN = "@Observable"  # Change this

results = bash(f"grep -rn '{SEARCH_PATTERN}' --include='*.swift' {PROJECT_PATH} | head -20")
print(f"🔍 Search results for '{SEARCH_PATTERN}':")
print(results)

---
## 5️⃣ Edit a File

In [ ]:
# File to edit
FILE_TO_EDIT = "ContentView.swift"  # Change this

# Read current content
current_content = file_read(FILE_TO_EDIT)
print(f"📄 Current content of {FILE_TO_EDIT}:")
print(f"Length: {len(current_content)} chars")

In [ ]:
# Modify and write back
new_content = current_content  # Modify this!

# Example: Replace something
# new_content = current_content.replace("old_text", "new_text")

# Uncomment to write:
# result = file_write(FILE_TO_EDIT, new_content)
# print(result)

---
## 6️⃣ Build the Project

In [ ]:
# Find the .xcodeproj
xcode_files = bash(f"find {PROJECT_PATH}/.. -maxdepth 2 -name '*.xcodeproj' 2>/dev/null | head -5")
print("📱 Xcode project files:")
print(xcode_files)

In [ ]:
# Build the project
XCODEPROJ = f"{PROJECT_PATH}/../3-2-1-Liftoff.xcodeproj"  # Change this
SCHEME = "3-2-1-Liftoff"  # Change this

print(f"🔨 Building {SCHEME}...")
build_output = bash(f"xcodebuild -project '{XCODEPROJ}' -scheme '{SCHEME}' -destination 'platform=iOS Simulator,name=iPhone 16 Pro' build 2>&1 | tail -30", timeout=300)
print(build_output)

if "BUILD SUCCEEDED" in build_output:
    print("\n✅ BUILD SUCCEEDED")
elif "BUILD FAILED" in build_output:
    print("\n❌ BUILD FAILED")

---
## 7️⃣ Run in Simulator

In [ ]:
# Boot simulator
SIMULATOR_NAME = "iPhone 16 Pro"
bash(f"xcrun simctl boot '{SIMULATOR_NAME}' 2>/dev/null || true")
bash("open -a Simulator")
print(f"✅ {SIMULATOR_NAME} launched")

---
## 8️⃣ Take Simulator Screenshot

In [ ]:
# Take and display screenshot
screenshot()

---
## 9️⃣ Git Operations

In [ ]:
# Check git status
print(bash("git status"))

In [ ]:
# Commit changes
# bash("git add -A && git commit -m 'Update from notebook'")

---
## 🔄 Quick Iteration Template

In [ ]:
# === COPY THIS FOR EACH FIX ===
# 1. Read
target = "path/to/file.swift"
content = file_read(target)

# 2. Fix
fixed = content  # modify

# 3. Write
# file_write(target, fixed)

# 4. Build
# bash(f"xcodebuild -project '{XCODEPROJ}' -scheme '{SCHEME}' build 2>&1 | tail -10")